In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_11_try_0.pickle

In [ ]:
%%PandasProfile
### cell 12 ###

def bar_chart_multiple_choice_24(response_counts, title, y_axis_title, orientation='h', num_choices=15):
    # ensure a Series and set index name
    if not isinstance(response_counts, pd.Series):
        response_counts = pd.Series(response_counts)
    response_counts.index.name = ''
    # save to CSV in one step
    base_path = '/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/kaggle/working/individual_charts/data'
    response_counts.rename_axis('').to_frame().to_csv(f'{base_path}/{title}.csv')
    # compute scale (unused) for consistent behavior
    _ = response_counts[:num_choices].mul(1.2).max()

def count_then_return_percent_24(dataframe, x_axis_title):
    # use normalize to compute percentages directly
    return (
        dataframe[x_axis_title]
        .value_counts(dropna=False, normalize=True)
        .mul(100)
        .round(1)
    )

# set up question and country columns once
data_col = 'Are you currently a student? (high school, university, or graduate)'
country_col = 'In which country do you currently reside?'
# filter once for both countries
filtered = responses_df_2022_cell10.loc[
    responses_df_2022_cell10[country_col].isin(['United States of America', 'India']),
    [country_col, data_col]
]
# compute grouped percentages in one pass
grouped = (
    filtered
    .groupby(country_col)[data_col]
    .value_counts(dropna=False, normalize=True)
    .mul(100)
    .round(1)
)
# extract and sort per country
usa_percentages = grouped['United States of America'].sort_index()
india_percentages = grouped['India'].sort_index()

# generate CSV and chart for USA
bar_chart_multiple_choice_24(
    response_counts=usa_percentages,
    title='Students status for Kaggle Survey participants from the USA',
    y_axis_title='% of respondents',
    orientation=orientation_for_chart
)
# display India percentages info
india_percentages.info()